In [ ]:
"""
=====================================================================================
Hetero-GraphSAGE για Ανίχνευση Ανακληθέντων Άρθρων
=====================================================================================
Επιλογή πειράματος:

  'baseline'   → Πείραμα 0: Βασικό μοντέλο (paper + author)
  'cosine'     → Πείραμα 1: Σημασιολογικές ακμές (cosine similarity)
  'university' → Πείραμα 2: Κόμβοι-πανεπιστήμια
  'country'    → Πείραμα 3: Κόμβοι-χώρες
  'journal'    → Πείραμα 4: Κόμβοι-περιοδικά
  'virtual'    → Πείραμα 5: Εικονικός κόμβος (virtual node)
  'inductive'  → Πείραμα 6: Επαγωγική μάθηση (NeighborLoader)
=====================================================================================
"""

# =====================================================================================
# ΕΠΙΛΟΓΗ ΠΕΙΡΑΜΑΤΟΣ
# =====================================================================================
EXPERIMENT = 'baseline'

# =====================================================================================
# IMPORTS
# =====================================================================================
import os
import glob
import random
import re
import ast
from itertools import combinations
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc, precision_recall_curve, average_precision_score,
)

from torch_geometric.data import HeteroData
from torch_geometric.nn import SAGEConv, to_hetero
from torch.optim.lr_scheduler import ReduceLROnPlateau

# NeighborLoader μόνο για inductive
if EXPERIMENT == 'inductive':
    from torch_geometric.loader import NeighborLoader


# =====================================================================================
# ΠΑΡΑΜΕΤΡΟΙ
# =====================================================================================
SEED       = 123
LR         = 0.01 if EXPERIMENT != 'inductive' else 0.001
HIDDEN_DIM = 64
DROPOUT    = 0.5
MAX_EPOCHS = 150
PATIENCE   = 20
TOP_K_BIB  = 5
OUTPUT_DIR = '../results'
os.makedirs(OUTPUT_DIR, exist_ok=True)  # Colab: '/content/drive/MyDrive/Thesis/results'

os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEP = "=" * 55
print(f"[ΠΕΙΡΑΜΑ: {EXPERIMENT.upper()}]  Seed={SEED}  Συσκευή={device}")


# =====================================================================================
# ΒΟΗΘΗΤΙΚΕΣ ΣΥΝΑΡΤΗΣΕΙΣ
# =====================================================================================
def find_input_file(filename):
    """Αναζητά αρχείο στο /kaggle/input/ (ή Drive)."""
    local = os.path.join('..', 'data', filename)
    if os.path.exists(local):
        return local
    matches = glob.glob(f'/kaggle/input/**/{filename}', recursive=True)
    if not matches:
        # Fallback για Colab
        colab_path = f'/content/drive/MyDrive/Thesis/{filename}'
        if os.path.exists(colab_path):
            return colab_path
        raise FileNotFoundError(f"Δεν βρέθηκε '{filename}'.")
    return matches[0]


def normalize_id(raw_id) -> str:
    """Ομογενοποίηση OpenAlex URLs → καθαρά IDs (π.χ. W2741809807)."""
    s = str(raw_id).strip()
    s = s.replace('https://openalex.org/', '').replace('http://openalex.org/', '')
    return s.upper()


def get_time_features(row):
    """Εξάγει 3 χρονικά χαρακτηριστικά: [Max Yearly, Active Years, Burst Ratio]."""
    c_str       = str(row['counts_by_year'])
    total_cites = float(row['cited_by_count']) if pd.notna(row['cited_by_count']) else 0.0
    if c_str in ('nan', '[]', 'None', ''):
        return [0.0, 0.0, 0.0]
    try:
        y_cites = [item['cited_by_count'] for item in ast.literal_eval(c_str)]
        max_y   = float(max(y_cites))
        return [max_y, float(len(y_cites)), max_y / total_cites if total_cites > 0 else 0.0]
    except Exception:
        return [0.0, 0.0, 0.0]


# =====================================================================================
# ΦΟΡΤΩΣΗ DATASET & EMBEDDINGS
# =====================================================================================
print(f"\n{SEP}\n  ΦΟΡΤΩΣΗ DATASET\n{SEP}")

DATASET_CSV   = find_input_file('dataset_latest.csv')
EMBEDDINGS_PT = find_input_file('specter_embeddings_latest.pt')

df = pd.read_csv(DATASET_CSV).reset_index(drop=True)
abstract_embeddings = torch.load(EMBEDDINGS_PT, map_location='cpu')

assert len(df) == abstract_embeddings.shape[0], "Πλήθος γραμμών != πλήθος embeddings."
print(f"  Φορτώθηκαν {len(df)} άρθρα.")


# =====================================================================================
# STRATIFIED SPLIT  70 / 10 / 20
# =====================================================================================
indices = np.arange(len(df))
labels  = df['label'].values

train_idx, temp_idx, _, temp_labels = train_test_split(
    indices, labels, test_size=0.30, stratify=labels, random_state=SEED
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=(2/3), stratify=temp_labels, random_state=SEED
)
train_set = set(train_idx.tolist())

print(f"  Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")


# =====================================================================================
# ΔΟΜΗΣΗ ΓΡΑΦΟΥ — ΚΟΙΝΕΣ ΑΚΜΕΣ
# =====================================================================================
print(f"\n{SEP}\n  ΔΟΜΗΣΗ ΓΡΑΦΟΥ\n{SEP}")

writes_src,   writes_dst   = [], []
cites_src,    cites_dst    = [], []
coauthor_src, coauthor_dst = [], []
coref_src,    coref_dst    = [], []

author_mapping   = {}
author_has_orcid = []
author_info_list = []   # "university_country" ανά author
author_counter   = 0

paper_mapping       = {normalize_id(row['id']): i for i, row in df.iterrows()}
total_paper_counter = len(paper_mapping)
paper_to_authors    = defaultdict(list)
reference_to_papers = defaultdict(list)

# Για inductive: παρακολούθηση author membership
author_in_train = set()
author_in_nontr = set()

# Για Πείραμα 2 (university nodes)
university_mapping = {}
university_counter = 0
univ_src, univ_dst = [], []   # author → located_at → university

# Για Πείραμα 3 (country nodes)
country_mapping = {}
country_counter = 0
location_src, location_dst = [], []   # author → located_in → country

# Για Πείραμα 4 (journal nodes)
journal_mapping = {}
journal_counter = 0
published_src, published_dst = [], []   # paper → published_in → journal

# Για Πείραμα 5 (virtual node)
vnode_src, vnode_dst = [], []   # virtual_node → broadcasts_to → paper

pattern = r"([^()]+)\(([^)]+)\)"

for index, row in df.iterrows():
    p_idx = index

    # --- Α. Συγγραφείς & ORCID ---
    paper_orcids = []
    if 'orcid' in df.columns and pd.notna(row['orcid']):
        paper_orcids = [o.strip() for o in str(row['orcid']).split(';')]

    if pd.notna(row['authors']):
        matches = re.findall(pattern, str(row['authors']))
        orcids_reliable = (len(paper_orcids) == len(matches))

        for i, match in enumerate(matches):
            name      = match[0].replace(';', '').strip()
            affil_raw = match[1].strip()

            if ',' in affil_raw:
                university, country = affil_raw.rsplit(',', 1)
                university, country = university.strip(), country.strip()
            else:
                university, country = affil_raw, 'Unknown'

            has_valid_orcid = 0.0
            if orcids_reliable and i < len(paper_orcids):
                cur = paper_orcids[i]
                if cur and 'None' not in cur and len(cur) > 5:
                    has_valid_orcid = 1.0

            if name not in author_mapping:
                author_mapping[name] = author_counter
                author_has_orcid.append(has_valid_orcid)
                author_info_list.append(f"{university}_{country}")
                author_counter += 1
            else:
                if has_valid_orcid == 1.0:
                    author_has_orcid[author_mapping[name]] = 1.0

            a_id = author_mapping[name]
            writes_src.append(a_id)
            writes_dst.append(p_idx)
            paper_to_authors[p_idx].append(a_id)

            # Inductive: καταγραφή σε ποιο split ανήκει ο author
            if p_idx in train_set:
                author_in_train.add(a_id)
            else:
                author_in_nontr.add(a_id)

            # Πείραμα 2: σύνδεση author → university
            if EXPERIMENT == 'university' and university and university.lower() not in ('none', 'unknown', ''):
                if university not in university_mapping:
                    university_mapping[university] = university_counter
                    university_counter += 1
                univ_src.append(a_id)
                univ_dst.append(university_mapping[university])

            # Πείραμα 3: σύνδεση author → country
            if EXPERIMENT == 'country' and country and len(country) >= 2 and country.lower() not in ('none', 'unknown'):
                if country not in country_mapping:
                    country_mapping[country] = country_counter
                    country_counter += 1
                location_src.append(a_id)
                location_dst.append(country_mapping[country])

    # --- Β. Αναφορές ---
    if pd.notna(row['references']):
        for ref in str(row['references']).split(';'):
            ref = normalize_id(ref)
            if not ref:
                continue
            if ref not in paper_mapping:
                paper_mapping[ref] = total_paper_counter
                total_paper_counter += 1
            cites_src.append(p_idx)
            cites_dst.append(paper_mapping[ref])
            reference_to_papers[paper_mapping[ref]].append(p_idx)

    # --- Γ. Περιοδικά (Πείραμα 4) ---
    if EXPERIMENT == 'journal':
        journal_name = str(row['journal']).strip() if pd.notna(row.get('journal')) else 'Unknown'
        if journal_name not in journal_mapping:
            journal_mapping[journal_name] = journal_counter
            journal_counter += 1
        published_src.append(p_idx)
        published_dst.append(journal_mapping[journal_name])

    # --- Δ. Virtual Node (Πείραμα 5) ---
    if EXPERIMENT == 'virtual':
        vnode_src.append(0)
        vnode_dst.append(p_idx)

# --- Ε. Συν-συγγραφείς ---
for p, authors in paper_to_authors.items():
    if len(authors) > 1:
        for a1, a2 in combinations(authors, 2):
            coauthor_src.extend([a1, a2])
            coauthor_dst.extend([a2, a1])

# --- ΣΤ. Βιβλιογραφική Σύζευξη ---
# Inductive: fit μόνο σε train — χωρίς leakage στο val/test
if EXPERIMENT == 'inductive':
    train_refs  = df.loc[train_idx, 'references'].fillna('').apply(
        lambda s: ' '.join(normalize_id(r) for r in str(s).split(';') if r.strip())
    )
    vectorizer  = CountVectorizer(token_pattern=r'\S+')
    A_train     = vectorizer.fit_transform(train_refs)
    B_train     = A_train.dot(A_train.T)
    B_train.setdiag(0)
    bib_matrix  = torch.tensor(B_train.toarray(), dtype=torch.float32)
    bib_weights, bib_indices = torch.topk(bib_matrix, min(TOP_K_BIB, bib_matrix.shape[1]), dim=1)
    for local_i, global_i in enumerate(train_idx):
        for j in range(bib_weights.shape[1]):
            if bib_weights[local_i, j].item() >= 2.0:
                global_j = train_idx[bib_indices[local_i, j].item()]
                coref_src.append(int(global_i))
                coref_dst.append(int(global_j))
else:
    # Transductive: fit σε όλα τα άρθρα (αποδεκτό — labels test hidden)
    ref_strings = df['references'].fillna('').apply(
        lambda s: ' '.join(normalize_id(r) for r in str(s).split(';') if r.strip())
    )
    vectorizer  = CountVectorizer(token_pattern=r'\S+')
    A           = vectorizer.fit_transform(ref_strings)
    B           = A.dot(A.T).tolil()
    B.setdiag(0)
    B           = B.tocsr()
    bib_matrix  = torch.tensor(B.toarray(), dtype=torch.float32)
    bib_weights, bib_indices = torch.topk(bib_matrix, TOP_K_BIB, dim=1)
    for i in range(len(df)):
        for j in range(TOP_K_BIB):
            if bib_weights[i, j].item() >= 2.0:
                coref_src.append(i)
                coref_dst.append(bib_indices[i, j].item())

print(f"  writes: {len(writes_src)} | cites: {len(cites_src)}")
print(f"  co-authorship: {len(coauthor_src)} | bib-coupling: {len(coref_src)}")


# =====================================================================================
# ① ΣΤΑΤΙΣΤΙΚΑ ΓΡΑΦΟΥ
# =====================================================================================
print(f"\n{SEP}")
print("  ΣΤΑΤΙΣΤΙΚΑ ΓΡΑΦΟΥ")
print(SEP)
print(f"  {'ΚΟΜΒΟΙ':40s} {'ΠΛΗΘΟΣ':>8}")
print(f"  {'-'*48}")
print(f"  {'paper  (dataset)':40s} {len(df):>8,}")
print(f"  {'paper  (+ leaf references)':40s} {total_paper_counter:>8,}")
print(f"  {'author':40s} {author_counter:>8,}")
if EXPERIMENT == 'university':
    print(f"  {'university':40s} {university_counter:>8,}")
if EXPERIMENT == 'country':
    print(f"  {'country':40s} {country_counter:>8,}")
if EXPERIMENT == 'journal':
    print(f"  {'journal':40s} {journal_counter:>8,}")
if EXPERIMENT == 'virtual':
    print(f"  {'virtual_node':40s} {1:>8,}")

print(f"\n  {'ΑΚΜΕΣ (ανά τύπο)':40s} {'ΠΛΗΘΟΣ':>8}")
print(f"  {'-'*48}")
print(f"  {'author → writes → paper':40s} {len(writes_src):>8,}")
print(f"  {'paper  → written_by → author':40s} {len(writes_src):>8,}")
print(f"  {'paper  → cites → paper':40s} {len(cites_src):>8,}")
print(f"  {'paper  → cited_by → paper':40s} {len(cites_src):>8,}")
print(f"  {'author → collaborates → author':40s} {len(coauthor_src):>8,}")
print(f"  {'paper  → shares_refs_with → paper':40s} {len(coref_src):>8,}")
print(f"  {'paper  → shared_refs_by → paper':40s} {len(coref_src):>8,}")
if EXPERIMENT == 'university':
    print(f"  {'author → located_at → university':40s} {len(univ_src):>8,}")
    print(f"  {'university → has_author → author':40s} {len(univ_src):>8,}")
if EXPERIMENT == 'country':
    print(f"  {'author → located_in → country':40s} {len(location_src):>8,}")
    print(f"  {'country → has_author → author':40s} {len(location_src):>8,}")
if EXPERIMENT == 'journal':
    print(f"  {'paper  → published_in → journal':40s} {len(published_src):>8,}")
    print(f"  {'journal → has_paper → paper':40s} {len(published_src):>8,}")
if EXPERIMENT == 'virtual':
    print(f"  {'virtual_node → broadcasts_to → paper':40s} {len(vnode_src):>8,}")
    print(f"  {'paper → sends_to → virtual_node':40s} {len(vnode_dst):>8,}")
if EXPERIMENT == 'inductive':
    print(f"  {'  (bib coupling: train papers μόνο)':40s}")
print(SEP)


# =====================================================================================
# ΧΑΡΑΚΤΗΡΙΣΤΙΚΑ ΚΟΜΒΩΝ (FEATURES)
# =====================================================================================
percentile_tensor = torch.tensor(
    df['cited_by_percentile_year'].fillna(0).values / 100.0, dtype=torch.float32
).unsqueeze(1)

count_tensor = torch.log1p(
    torch.tensor(df['cited_by_count'].fillna(0).values, dtype=torch.float32)
).unsqueeze(1)

time_tensor    = torch.tensor(df.apply(get_time_features, axis=1).tolist(), dtype=torch.float32)
# Κανονικοποίηση: max από train μόνο — αποφυγή data leakage σε val/test
train_max_vals = time_tensor[train_idx].max(dim=0, keepdim=True)[0]
time_tensor    = torch.where(train_max_vals > 0, time_tensor / train_max_vals, time_tensor)

embeddings_norm = F.normalize(abstract_embeddings, p=2, dim=1)
paper_features  = torch.cat([embeddings_norm, percentile_tensor, count_tensor, time_tensor], dim=1)
paper_labels    = torch.tensor(df['label'].values, dtype=torch.long)

# Leaf nodes (εξωτερικές αναφορές) παίρνουν μηδενικά features και label=-1
if total_paper_counter > len(df):
    extra = total_paper_counter - len(df)
    paper_features = torch.cat([paper_features, torch.zeros((extra, paper_features.shape[1]))], dim=0)
    paper_labels   = torch.cat([paper_labels, torch.full((extra,), -1, dtype=torch.long)], dim=0)

author_features_tensor = (
    torch.tensor(author_has_orcid, dtype=torch.float32).unsqueeze(1)
    if author_has_orcid else
    torch.zeros((author_counter, 1), dtype=torch.float32)
)

# Πείραμα 1: cosine similarity embeddings για κατασκευή ακμών
if EXPERIMENT == 'cosine':
    from sklearn.metrics.pairwise import cosine_similarity as sk_cosine
    TOP_K_COS  = 5
    emb_np     = embeddings_norm.numpy()
    cos_matrix = sk_cosine(emb_np)
    np.fill_diagonal(cos_matrix, 0.0)
    cosine_src, cosine_dst = [], []
    for i in range(len(df)):
        top_k = np.argsort(cos_matrix[i])[::-1][:TOP_K_COS]
        for j in top_k:
            if cos_matrix[i, j] >= 0.85:   # κατώφλι ομοιότητας
                cosine_src.append(i)
                cosine_dst.append(j)
    print(f"  Cosine similarity ακμές (≥0.85): {len(cosine_src)}")


# =====================================================================================
# ΣΥΝΑΡΜΟΛΟΓΗΣΗ HeteroData
# =====================================================================================
data = HeteroData()
data['paper'].x  = paper_features
data['paper'].y  = paper_labels
data['author'].x = author_features_tensor

# Dummy features για κόμβους χωρίς κειμενική πληροφορία
if EXPERIMENT == 'university' and university_counter > 0:
    data['university'].x = torch.ones((university_counter, 1), dtype=torch.float32)
if EXPERIMENT == 'country' and country_counter > 0:
    data['country'].x = torch.ones((country_counter, 1), dtype=torch.float32)
if EXPERIMENT == 'journal' and journal_counter > 0:
    data['journal'].x = torch.ones((journal_counter, 1), dtype=torch.float32)
if EXPERIMENT == 'virtual':
    # Ο εικονικός κόμβος έχει μόνο 1 node — LayerNorm αντί BatchNorm
    data['virtual_node'].x = torch.ones((1, 1), dtype=torch.float32)

# Κοινές ακμές
if writes_src:
    data['author', 'writes',     'paper'].edge_index = torch.tensor([writes_src, writes_dst], dtype=torch.long)
    data['paper',  'written_by', 'author'].edge_index = torch.tensor([writes_dst, writes_src], dtype=torch.long)
if cites_src:
    data['paper', 'cites',    'paper'].edge_index = torch.tensor([cites_src, cites_dst], dtype=torch.long)
    data['paper', 'cited_by', 'paper'].edge_index = torch.tensor([cites_dst, cites_src], dtype=torch.long)
if coauthor_src:
    data['author', 'collaborates_with', 'author'].edge_index = torch.tensor([coauthor_src, coauthor_dst], dtype=torch.long)
if coref_src:
    data['paper', 'shares_refs_with', 'paper'].edge_index = torch.tensor([coref_src, coref_dst], dtype=torch.long)
    data['paper', 'shared_refs_by',   'paper'].edge_index = torch.tensor([coref_dst, coref_src], dtype=torch.long)

# Πείραμα-συγκεκριμένες ακμές
if EXPERIMENT == 'cosine' and cosine_src:
    data['paper', 'similar_to',      'paper'].edge_index = torch.tensor([cosine_src, cosine_dst], dtype=torch.long)
    data['paper', 'similar_to_rev',  'paper'].edge_index = torch.tensor([cosine_dst, cosine_src], dtype=torch.long)
if EXPERIMENT == 'university' and univ_src:
    data['author',      'located_at',  'university'].edge_index = torch.tensor([univ_src, univ_dst], dtype=torch.long)
    data['university',  'has_author',  'author'].edge_index     = torch.tensor([univ_dst, univ_src], dtype=torch.long)
if EXPERIMENT == 'country' and location_src:
    data['author',  'located_in', 'country'].edge_index = torch.tensor([location_src, location_dst], dtype=torch.long)
    data['country', 'has_author', 'author'].edge_index  = torch.tensor([location_dst, location_src], dtype=torch.long)
if EXPERIMENT == 'journal' and published_src:
    data['paper',   'published_in', 'journal'].edge_index = torch.tensor([published_src, published_dst], dtype=torch.long)
    data['journal', 'has_paper',    'paper'].edge_index   = torch.tensor([published_dst, published_src], dtype=torch.long)
if EXPERIMENT == 'virtual' and vnode_src:
    data['virtual_node', 'broadcasts_to', 'paper'].edge_index = torch.tensor([vnode_src, vnode_dst], dtype=torch.long)
    data['paper',        'sends_to', 'virtual_node'].edge_index = torch.tensor([vnode_dst, vnode_src], dtype=torch.long)

# Masks
data['paper'].train_mask = torch.zeros(total_paper_counter, dtype=torch.bool)
data['paper'].val_mask   = torch.zeros(total_paper_counter, dtype=torch.bool)
data['paper'].test_mask  = torch.zeros(total_paper_counter, dtype=torch.bool)
data['paper'].train_mask[train_idx] = True
data['paper'].val_mask[val_idx]     = True
data['paper'].test_mask[test_idx]   = True


# =====================================================================================
# ΜΟΝΤΕΛΟ
# =====================================================================================
class RetractionDetectorSAGE(torch.nn.Module):
    """2-layer Hetero-GraphSAGE με BatchNorm (ή LayerNorm για virtual node)."""
    def __init__(self, hidden_dim: int, dropout_p: float, use_layer_norm: bool = False):
        super().__init__()
        self.conv1    = SAGEConv((-1, -1), hidden_dim)
        self.dropout  = torch.nn.Dropout(p=dropout_p)
        self.conv2    = SAGEConv((-1, -1), 2)
        # LayerNorm για virtual node (1 node → BatchNorm αποτυγχάνει)
        if use_layer_norm:
            self.norm = torch.nn.LayerNorm(hidden_dim)
        else:
            self.norm = torch.nn.BatchNorm1d(hidden_dim)

    def forward(self, x, edge_index):
        x = F.relu(self.norm(self.conv1(x, edge_index)))
        x = self.dropout(x)
        return self.conv2(x, edge_index)


# Virtual node έχει 1 node → LayerNorm αναγκαίο
use_layer_norm = (EXPERIMENT == 'virtual')

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


# =====================================================================================
# ΕΚΠΑΙΔΕΥΣΗ — TRANSDUCTIVE (πειράματα 0-5)
# =====================================================================================
if EXPERIMENT != 'inductive':
    data = data.to(device)
    model     = to_hetero(RetractionDetectorSAGE(HIDDEN_DIM, DROPOUT, use_layer_norm),
                          data.metadata(), aggr='mean').to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

    train_mask = data['paper'].train_mask
    val_mask   = data['paper'].val_mask

    train_losses, val_losses, val_f1s = [], [], []
    best_val_loss    = float('inf')
    best_epoch       = 0
    best_state_dict  = None
    patience_counter = 0

    print(f"\n{SEP}")
    print("   ΕΚΠΑΙΔΕΥΣΗ  (Transductive / Full-Graph)")
    print(SEP)
    print(f"  {'Epoch':>6}  {'Train Loss':>10}  {'Val Loss':>10}  {'Val F1':>8}")
    print(f"  {'-'*40}")

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        optimizer.zero_grad()
        out        = model(data.x_dict, data.edge_index_dict)
        train_loss = F.cross_entropy(out['paper'][train_mask],
                                     data['paper'].y[train_mask], ignore_index=-1)
        train_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        model.eval()
        with torch.no_grad():
            out        = model(data.x_dict, data.edge_index_dict)
            val_logits = out['paper'][val_mask]
            val_loss   = F.cross_entropy(val_logits, data['paper'].y[val_mask], ignore_index=-1)
            val_pred   = val_logits.argmax(dim=-1).cpu().numpy()
            val_true   = data['paper'].y[val_mask].cpu().numpy()
            val_f1     = f1_score(val_true, val_pred, zero_division=0)

        scheduler.step(val_loss)
        train_losses.append(train_loss.item())
        val_losses.append(val_loss.item())
        val_f1s.append(val_f1)

        if val_loss.item() < best_val_loss:
            best_val_loss   = val_loss.item()
            best_epoch      = epoch
            patience_counter = 0
            best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1

        if epoch % 10 == 0:
            print(f"  {epoch:>6}  {train_loss.item():>10.4f}  {val_loss.item():>10.4f}  {val_f1:>8.4f}")

        if patience_counter >= PATIENCE:
            print(f"\n  Early stopping: epoch {epoch}  |  Best epoch: {best_epoch}")
            break

    print(f"  {'-'*40}")
    print(f"  Best checkpoint → Val Loss: {best_val_loss:.4f}")
    print(SEP)

    # --- Αξιολόγηση Test Set ---
    model.load_state_dict({k: v.to(device) for k, v in best_state_dict.items()})
    model.eval()

    with torch.no_grad():
        out         = model(data.x_dict, data.edge_index_dict)
        test_logits = out['paper'][data['paper'].test_mask]
        test_probs  = torch.softmax(test_logits, dim=-1)[:, 1].cpu().numpy()
        test_pred   = test_logits.argmax(dim=-1).cpu().numpy()
        test_true   = data['paper'].y[data['paper'].test_mask].cpu().numpy()


# =====================================================================================
# ΕΚΠΑΙΔΕΥΣΗ — INDUCTIVE (πείραμα 6)
# =====================================================================================
else:
    # Inductive subgraph: μόνο train papers + safe authors
    # Safe = authors που δεν εμφανίζονται σε val/test (αποφυγή structural leakage)
    safe_train_authors = author_in_train - author_in_nontr
    safe_author_mask   = torch.zeros(author_counter, dtype=torch.bool)
    for a_id in safe_train_authors:
        safe_author_mask[a_id] = True

    print(f"  Authors σε train (safe): {safe_author_mask.sum().item()}")
    print(f"  Authors κοινοί (εξαιρούνται): {len(author_in_nontr & author_in_train)}")

    train_data_subgraph = data.subgraph({
        'paper':  data['paper'].train_mask,
        'author': safe_author_mask
    })

    # Mini-batch loaders
    train_loader = NeighborLoader(
        train_data_subgraph,
        num_neighbors=[10, 10], batch_size=256,
        input_nodes=('paper', train_data_subgraph['paper'].train_mask),
        shuffle=True
    )
    val_loader = NeighborLoader(
        data, num_neighbors=[10, 10], batch_size=256,
        input_nodes=('paper', data['paper'].val_mask)
    )
    test_loader = NeighborLoader(
        data, num_neighbors=[10, 10], batch_size=256,
        input_nodes=('paper', data['paper'].test_mask)
    )

    data = data.to(device)
    model     = to_hetero(RetractionDetectorSAGE(HIDDEN_DIM, DROPOUT),
                          data.metadata(), aggr='mean').to(device)
    # Inductive: weight_decay για καλύτερη γενίκευση
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=5e-4)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

    train_losses, val_losses, val_f1s = [], [], []
    best_val_loss    = float('inf')
    best_epoch       = 0
    best_state_dict  = None
    patience_counter = 0

    print(f"\n{SEP}")
    print("  ΕΚΠΑΙΔΕΥΣΗ  (Inductive / Mini-Batch NeighborLoader)")
    print(SEP)
    print(f"  {'Epoch':>6}  {'Train Loss':>10}  {'Val Loss':>10}  {'Val F1':>8}")
    print(f"  {'-'*40}")

    for epoch in range(1, MAX_EPOCHS + 1):
        # --- Train (mini-batches) ---
        model.train()
        total_loss, total_batches = 0.0, 0
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            out  = model(batch.x_dict, batch.edge_index_dict)
            mask = batch['paper'].train_mask
            loss = F.cross_entropy(out['paper'][mask], batch['paper'].y[mask], ignore_index=-1)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss    += loss.item()
            total_batches += 1
        avg_train_loss = total_loss / max(total_batches, 1)

        # --- Validation (mini-batches) ---
        model.eval()
        total_val_loss, total_val_batches = 0.0, 0
        val_preds_all, val_trues_all = [], []
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                out   = model(batch.x_dict, batch.edge_index_dict)
                mask  = batch['paper'].val_mask
                total_val_loss    += F.cross_entropy(out['paper'][mask], batch['paper'].y[mask], ignore_index=-1).item()
                total_val_batches += 1
                val_preds_all.extend(out['paper'][mask].argmax(dim=-1).cpu().numpy())
                val_trues_all.extend(batch['paper'].y[mask].cpu().numpy())
        avg_val_loss = total_val_loss / max(total_val_batches, 1)
        val_f1       = f1_score(val_trues_all, val_preds_all, zero_division=0)

        scheduler.step(avg_val_loss)
        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)
        val_f1s.append(val_f1)

        if avg_val_loss < best_val_loss:
            best_val_loss   = avg_val_loss
            best_epoch      = epoch
            patience_counter = 0
            best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1

        if epoch % 10 == 0:
            print(f"  {epoch:>6}  {avg_train_loss:>10.4f}  {avg_val_loss:>10.4f}  {val_f1:>8.4f}")

        if patience_counter >= PATIENCE:
            print(f"\n  Early stopping: epoch {epoch}  |  Best epoch: {best_epoch}")
            break

    print(f"  {'-'*40}")
    print(f"  Best checkpoint → Val Loss: {best_val_loss:.4f}")
    print(SEP)

    # --- Αξιολόγηση Test Set (με deduplication λόγω NeighborLoader overlap) ---
    model.load_state_dict({k: v.to(device) for k, v in best_state_dict.items()})
    model.eval()

    all_preds, all_trues, all_probs, all_ids = [], [], [], []
    with torch.no_grad():
        for batch in test_loader:
            batch  = batch.to(device)
            out    = model(batch.x_dict, batch.edge_index_dict)
            mask   = batch['paper'].test_mask
            logits = out['paper'][mask]
            ids    = batch['paper'].n_id[mask].cpu().numpy()
            all_probs.extend(torch.softmax(logits, dim=-1)[:, 1].cpu().numpy())
            all_preds.extend(logits.argmax(dim=-1).cpu().numpy())
            all_trues.extend(batch['paper'].y[mask].cpu().numpy())
            all_ids.extend(ids)

    # Αφαίρεση διπλοτύπων: ο NeighborLoader επιστρέφει ίδιους κόμβους σε διαφορετικά batches
    seen = set()
    unique_preds, unique_trues, unique_probs = [], [], []
    for i, nid in enumerate(all_ids):
        if nid not in seen:
            seen.add(nid)
            unique_preds.append(all_preds[i])
            unique_trues.append(all_trues[i])
            unique_probs.append(all_probs[i])

    test_pred  = np.array(unique_preds)
    test_true  = np.array(unique_trues)
    test_probs = np.array(unique_probs)
    print(f"  Test κόμβοι μετά deduplication: {len(test_pred)}")


# =====================================================================================
# ΑΠΟΤΕΛΕΣΜΑΤΑ TEST SET
# =====================================================================================
test_acc  = accuracy_score(test_true, test_pred)
test_prec = precision_score(test_true, test_pred, zero_division=0)
test_rec  = recall_score(test_true, test_pred, zero_division=0)
test_f1   = f1_score(test_true, test_pred, zero_division=0)

fpr, tpr, _                      = roc_curve(test_true, test_probs)
roc_auc                           = auc(fpr, tpr)
precision_curve, recall_curve, _ = precision_recall_curve(test_true, test_probs)
ap                                = average_precision_score(test_true, test_probs)
cm_matrix                         = confusion_matrix(test_true, test_pred)

print(f"\n{SEP}")
print("  ΑΠΟΤΕΛΕΣΜΑΤΑ TEST SET")
print(SEP)
print(f"  {'Μετρική':30s} {'Τιμή':>10}")
print(f"  {'-'*40}")
print(f"  {'Accuracy':30s} {test_acc:>10.4f}")
print(f"  {'Precision':30s} {test_prec:>10.4f}")
print(f"  {'Recall':30s} {test_rec:>10.4f}")
print(f"  {'F1 Score':30s} {test_f1:>10.4f}")
print(f"  {'ROC AUC':30s} {roc_auc:>10.4f}")
print(f"  {'Average Precision (PR)':30s} {ap:>10.4f}")
print(SEP)


# =====================================================================================
# ΓΡΑΦΗΜΑΤΑ
# =====================================================================================
exp_tag = EXPERIMENT
epochs_range = range(1, len(train_losses) + 1)

# --- Train/Val Loss + Val F1 ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(epochs_range, train_losses, label='Train Loss', color='steelblue')
axes[0].plot(epochs_range, val_losses,   label='Val Loss',   color='darkorange')
axes[0].axvline(best_epoch, color='red', linestyle='--', label=f'Best epoch ({best_epoch})')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Train / Val Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(epochs_range, val_f1s, label='Val F1', color='green')
axes[1].axvline(best_epoch, color='red', linestyle='--', label=f'Best epoch ({best_epoch})')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('F1 Score')
axes[1].set_title('Val F1 Score'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/loss_curve_{exp_tag}.png', dpi=150)
plt.close()

# --- Confusion Matrix ---
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm_matrix, display_labels=['Non-Retracted', 'Retracted']).plot(ax=ax, cmap='Blues')
ax.set_title('Confusion Matrix (Test Set)')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/confusion_matrix_{exp_tag}.png', dpi=150)
plt.close()

# --- ROC Curve ---
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.4f}')
ax.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--')
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.05])
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve (Test Set)'); ax.legend(loc='lower right'); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/roc_curve_{exp_tag}.png', dpi=150)
plt.close()

# --- Precision-Recall Curve ---
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(recall_curve, precision_curve, color='steelblue', lw=2, label=f'AP = {ap:.4f}')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve (Test Set)'); ax.legend(loc='upper right'); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/pr_curve_{exp_tag}.png', dpi=150)
plt.close()

print(f"\nΑποθηκεύτηκαν στο {OUTPUT_DIR}/:")
print(f"  loss_curve_{exp_tag}.png  |  confusion_matrix_{exp_tag}.png")
print(f"  roc_curve_{exp_tag}.png   |  pr_curve_{exp_tag}.png")
